In [1]:
from transformers import PretrainedConfig, PreTrainedModel, AutoModel, AutoFeatureExtractor
from transformers.models.qwen3.configuration_qwen3 import Qwen3Config
from transformers.modeling_outputs import ModelOutput
from typing import Optional, Union
import torch
import torch.nn as nn
import os
from dataclasses import dataclass
from transformers import AutoConfig

In [2]:
class OmniVoiceConfig(PretrainedConfig):
    model_type = "omnivoice"
    sub_configs = {"llm_config": Qwen3Config}

    def __init__(
        self,
        audio_vocab_size: int = 1025,
        audio_mask_id: int = 1024,
        num_audio_codebook: int = 8,
        audio_codebook_weights: Optional[list[float]] = None,
        llm_config: Optional[Union[dict, Qwen3Config]] = None,
        **kwargs,
    ):
        llm_config = Qwen3Config(**llm_config)
        self.llm_config = llm_config
        
        super().__init__(**kwargs)
        self.audio_vocab_size = audio_vocab_size
        self.audio_mask_id = audio_mask_id
        self.num_audio_codebook = num_audio_codebook
        self.audio_codebook_weights = audio_codebook_weights

In [3]:
def _resolve_model_path(name_or_path: str) -> str:
    if os.path.isdir(name_or_path):
        return name_or_path
    from huggingface_hub import snapshot_download

    return snapshot_download(name_or_path)

In [4]:
model_path = _resolve_model_path("k2-fsa/OmniVoice")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

In [5]:
config = OmniVoiceConfig.from_pretrained(model_path)

In [6]:
audio_embeddings = nn.Embedding(
    config.num_audio_codebook * config.audio_vocab_size,
    config.llm_config.hidden_size,
)
audio_heads = nn.Linear(
    config.llm_config.hidden_size,
    config.num_audio_codebook * config.audio_vocab_size,
    bias=False,
)
llm = AutoModel.from_pretrained(
    "./omnivoice_qwen3",
    dtype=torch.float32,
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [7]:
audio_embeddings.load_state_dict(
    torch.load(
        "./omnivoice_audio_embeddings/model.pt",
        map_location="cpu",
        weights_only=True,
    )
)
audio_heads.load_state_dict(
    torch.load(
        "./omnivoice_audio_heads/model.pt",
        map_location="cpu",
        weights_only=True,
    )
)

audio_embeddings = audio_embeddings.to(
    device="cuda",
    dtype=llm.dtype if hasattr(llm, "dtype") else torch.float32,
)
audio_heads = audio_heads.to(
    device="cuda",
    dtype=torch.float32,
)

llm = llm.to("cuda")

codebook_layer_offsets = torch.arange(config.num_audio_codebook) * config.audio_vocab_size
codebook_layer_offsets = codebook_layer_offsets.to("cuda")

### Example Input

In [9]:
save_data = torch.load(
    "batch_voice_audio.pt",
    map_location="cuda:0",
    weights_only=False,
)

In [10]:
gen_config = save_data["gen_config"]
short_task = save_data["short_task"]
batch_input_ids = save_data["batch_input_ids"]
batch_audio_mask = save_data["batch_audio_mask"]
batch_attention_mask = save_data["batch_attention_mask"]
schedules = save_data["schedules"]
c_lens = save_data["c_lens"]
layer_ids = save_data["layer_ids"]
tokens = save_data["tokens"]

In [11]:
B = short_task.batch_size

In [12]:
input_ids=batch_input_ids
audio_mask=batch_audio_mask
attention_mask=batch_attention_mask

### LLM Embeddings

In [13]:
input_embeddings = llm.get_input_embeddings()

In [14]:
# innput llm embeddings
input_ids[:, 0, :].shape

torch.Size([2, 224])

In [15]:
# llm embeddings
text_embeds = input_embeddings(input_ids[:, 0, :])

In [16]:
# output embeddings
text_embeds.shape

torch.Size([2, 224, 1024])

### Audio Embeddings

In [17]:
shifted_ids = (
    input_ids * audio_mask.unsqueeze(1)
) + codebook_layer_offsets.view(1, -1, 1) 

In [18]:
# input audio_embeddings
shifted_ids.shape

torch.Size([2, 8, 224])

In [19]:
# audio embeddings
audio_embeds = audio_embeddings(shifted_ids).sum(dim=1)

In [20]:
# output audio embeddings
audio_embeds.shape

torch.Size([2, 224, 1024])

### LLM

In [21]:
inputs_embeds = torch.where(
    audio_mask.unsqueeze(-1), audio_embeds, text_embeds
)

In [22]:
# input llm
inputs_embeds.shape, attention_mask.shape

(torch.Size([2, 224, 1024]), torch.Size([2, 1, 224, 224]))

In [23]:
# llm
llm_outputs = llm(
    inputs_embeds=inputs_embeds,
    attention_mask=attention_mask,
    return_dict=True,
    position_ids=None,
)    

In [24]:
# output llm
hidden_states = llm_outputs[0]
hidden_states.shape

torch.Size([2, 224, 1024])

### Audio Heads

In [25]:
batch_size, seq_len, _ = hidden_states.shape

In [26]:
# input audio heads
hidden_states.shape

torch.Size([2, 224, 1024])

In [27]:
# audio heads
logits_flat = audio_heads(hidden_states)

In [28]:
# output audio heads
logits_flat.shape

torch.Size([2, 224, 8200])

### Last Operation

In [29]:
audio_logits = logits_flat.view(
    batch_size,
    seq_len,
    config.num_audio_codebook,
    config.audio_vocab_size,
).permute(0, 2, 1, 3) 

In [30]:
audio_logits.shape

torch.Size([2, 8, 224, 1025])